Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [5]:
!pip install -q langchain langchain-community langchain-text-splitters faiss-cpu pypdf sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# ===== STEP 1: IMPORTS =====
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

# ===== STEP 2: LOAD PDF =====
file_path = "/content/dbms.pdf"

loader = PyPDFLoader(file_path)
documents = loader.load()[:20]

print("✅ PDF loaded successfully")

# ===== STEP 3: SPLIT TEXT =====
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)
print("✅ Total chunks:", len(docs))

# ===== STEP 4: CREATE EMBEDDINGS =====
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)
print("✅ Embeddings created")

# ===== STEP 5: LOAD LLM ====

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)
# ===== STEP 6: FUNCTION TO GENERATE ANSWER =====
def generate_answer(context, query):
    result = qa_pipeline(
        question=query,
        context=context
    )
    return result["answer"]

while True:
    query = input("\n💬 Ask a question (type 'exit' to stop): ")

    if query.lower() == "exit":
        break

    print("⏳ Generating answer...")

    results = vectorstore.similarity_search(query, k=2)
    context = " ".join([doc.page_content for doc in results])

    answer = generate_answer(context, query)

    print("\n🤖 Answer:", answer)

✅ PDF loaded successfully
✅ Total chunks: 136


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings created


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



💬 Ask a question (type 'exit' to stop): what is dbms
⏳ Generating answer...

🤖 Answer: Database Management System
